In [1]:
import pyspark
print(pyspark.__version__)

4.0.1


In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .appName("PartitioningBucketingDemo") \
    .config("spark.sql.warehouse.dir", "/tmp/spark_warehouse") \
    .enableHiveSupport() \
    .getOrCreate()

In [4]:
spark

In [3]:
# Sample Data
data = [
    (1, "Alice", "HR", 5000),
    (2, "Bob", "IT", 6000),
    (3, "Cathy", "HR", 5500),
    (4, "David", "IT", 7000),
    (5, "Eva", "Finance", 6500),
    (6, "Frank", "Finance", 6200)
]

columns = ["id", "name", "dept", "salary"]
df = spark.createDataFrame(data, columns)

In [4]:
(
    df.write
      .mode("overwrite")
      .partitionBy("dept")
      .parquet("/tmp/partitioned_employees")
)

print("Partitioned data written to /tmp/partitioned_employees")

Py4JJavaError: An error occurred while calling o56.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 7 in stage 0.0 failed 1 times, most recent failure: Lost task 7.0 in stage 0.0 (TID 7) (192.168.1.80 executor driver): org.apache.spark.SparkException: Python worker failed to connect back.
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:252)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:143)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:158)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:178)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:261)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scala:70)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:374)
	at org.apache.spark.rdd.RDD.iterator(RDD.scala:338)
	at org.apache.spark.rdd.MapPartitionsRDD.compute(MapPartitionsRD

In [6]:
(
    df.write
      .mode("overwrite")
      .bucketBy(4, "id")
      .sortBy("id")
      .saveAsTable("bucketed_employees")
)

print("Bucketed table 'bucketed_employees' created")

Py4JJavaError: An error occurred while calling o70.saveAsTable.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 6 in stage 1.0 failed 1 times, most recent failure: Lost task 6.0 in stage 1.0 (TID 14) (192.168.1.80 executor driver): org.apache.spark.SparkException: Python worker failed to connect back.
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:252)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:143)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:158)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:178)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:261)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scala:70)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:374)
	at org.apache.spark.rdd.RDD.iterator(RDD.scala:338)
	at org.apache.spark.rdd.MapPartitionsRDD.compute(MapPartiti

In [ ]:
df_partitioned = spark.read.parquet("/tmp/partitioned_employees")
df_partitioned.show()

print()
print()

df_bucketed = spark.table("bucketed_employees")
df_bucketed.show()

In [ ]:
df_partitioned.filter(col("dept") == "IT").explain(True)

In [ ]:
spark.sql("SELECT * FROM bucketed_employees WHERE id = 4").explain(True)